# Photon-noise NEP fit

Fit NEP vs incident power to the photon-noise NEP to get the optical efficiency:

$$\mathrm{NEP}_\mathrm{photon} = \sqrt{2 h \nu P / \eta}$$

Both NEP and P must be referred to incident power at one frequency $\nu$. Only the
photon-noise dominated (high power) regime is fit: points below `p_min` are excluded.

The fit is least squares in log space (each point gives $\eta_i = 2h\nu P_i / \mathrm{NEP}_i^2$,
and $\eta$ is their geometric mean). This is the right weighting when every NEP point has the
same *fractional* uncertainty, as when Sxx and R each have a constant fractional
uncertainty. Pass `nep_errs` to weight points by known uncertainties instead.
`eta_err` is statistical only: a common responsivity error scales every NEP and is not included.

In [ ]:
import numpy as np
import zarr
from citkid.nep import fit_nep_photon, fit_nep_photon_sets
from citkid.nep.interactive import run_nep_fit

nu = 150e9  # photon frequency (Hz)

# One array of incident power (W) and one of NEP (W / Hz^0.5) per set (e.g. per
# resonator). NaN points are ignored; a set that is all NaN gets NaN fit values.
powers = [...]
neps = [...]
names = None  # optional set names, e.g. [f'res {i}' for i in range(len(neps))]

## One set, without the GUI

Fits return numpy arrays and don't plot; review fits in the interactive window below.

In [ ]:
eta, eta_err, mask, n_fit = fit_nep_photon(
    powers[0], neps[0], nu,
    p_min = 1e-12,   # minimum power of the photon-noise dominated regime (W)
)
print(f'eta = {eta:.3f} +/- {eta_err:.3f} from {n_fit} points')

## Interactive: choose p_min for each set and review the fits

- Drag the dashed line (or use the `p_min` box) to set the minimum fitted power; the fit updates at once.
- `←/A`, `→/D` previous / next set. With a single initial `p_min`, the value you choose carries over to sets you haven't visited yet.
- `B` marks the set bad (NaN results); press again to unmark. `Shift+A` applies this set's `p_min` to every set. `R` rescales.
- With `group`, each set's `p_min`, bad flag and results are saved, and the next session resumes at the first set you haven't viewed. A set counts as viewed once you leave it, or if it is open when you close the window.

In [ ]:
root = zarr.open(r'path\to\output.zarr', mode = 'a')

p_min, eta, eta_err, n_fit, bad = run_nep_fit(
    powers, neps, nu,
    p_min = 1e-12,                       # initial p_min: one value, or one per set
    names = names,
    group = root.require_group('nep_fit'),
    # nep_errs = [...],                  # optional per-point NEP uncertainties
)

## Batch fit without the GUI

In [ ]:
eta, eta_err, n_fit = fit_nep_photon_sets(powers, neps, nu, p_min = 1e-12)